In [1]:
import transformers

/workspace/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model and tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

# Hook to capture activation from the first layer
activations = {}

def hook_fn(module, input, output):
    activations['first_layer'] = output[0]  # output[0] is the hidden states

# Register hook on the first transformer layer
hook = model.transformer.h[0].register_forward_hook(hook_fn)

# Prepare input
prompt = "Hello world"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

# Forward pass
with torch.no_grad():
    outputs = model(input_ids)

# Remove the hook
hook.remove()

# Display the activation
print("Shape of activation from first layer:", activations['first_layer'].shape)
print("Activation values (first 5 tokens, first 10 dimensions):")
print(activations['first_layer'][0, :5, :10])